In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [75]:
from openai import OpenAI
import os

from openai import OpenAI
openai_client = OpenAI()

In [76]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [77]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s up with you?'

In [78]:
question = "I just discovered the course now. Can I join now ?"
answer=llm(question)
print(answer)

Yes — in many cases you can still join, but it depends on the course’s enrollment rules and whether registration is still open.

A good reply you can send is:

**“I just discovered the course now. Is it still possible for me to join?”**

If you want, I can also help you make it:
- more polite,
- more formal,
- or shorter for email/text.


In [79]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [80]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, you can join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [81]:
# RAG stands for Retrieval-Augmented Generation
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [82]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'''{url_prefix}/{course["path"]}'''

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print(documents[0])
len(documents)

{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}


1350

In [83]:
from minsearch import Index


index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [84]:
question = "I just discovered the course. Can I join now?"

In [85]:
def search(question, course="llm-zoomcamp"):
    boost_dict={"question": 2.0, "section": 0.5}
    filter_dict={"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)

In [86]:
def build_context(search_results):
    lines=[]

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: "+ doc["question"])
        lines.append("A: "+ doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

In [87]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [88]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [89]:
question = "I just discovered the course. Can I join now?"

prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [90]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [91]:
response.output_text

response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=44, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=524)

In [92]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000558

In [93]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [97]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history=[{"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
        
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [98]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(INSTRUCTIONS, prompt, model=model)

In [99]:
answer = rag("I just discovered the course. Can I join now?")

print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [100]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort** and **pass the Capstone project**.\n\nA few important notes:\n- **Self-paced mode does not provide certificates.**\n- You also need to **peer-review 3 capstones** after submitting your project.\n- Homework is **not mandatory** for the certificate, though it is recommended.\n\nIf you want your real name on the certificate, set your **official name** in your course profile.'